In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # 解决 Mac M4 内核崩溃问题
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'    # 限制服务器上只用 0,1 号卡

import sys
import numpy as np 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

batch_size = 16
lr = 1e-4
max_epochs = 100

# 2. 智能识别设备（此时服务器上的 cuda:0 实际对应的就是物理上的 0 号卡）
if torch.cuda.is_available():
    device = torch.device("cuda:0") 
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"代码自动识别并连接到了: {device}")


In [ ]:
import sys
import torch
print(sys.version)
print(sys.executable)
print(torch.__version__)

In [ ]:
x = torch.tensor([[1.0000001, 2, 3], [2, 3, 5.5]])
print(x.dtype)
print(x.ndim)
print(x.numel()) # how many numbers
print(x.shape)
print(x.device)

In [ ]:
digit = torch.tensor([[0.0, 0.2, 0.9], [0.7, 0.3, 0.0]])
print(digit)
print(digit.shape) # (2,3)
print(digit.ndim) # 2
print(digit.numel()) # 6
print(digit.dtype) # float32
print(digit.device) #cpu

In [ ]:
"""
取出左下角的 0.7。
取出第1行的全部内容。
取出第2列的全部内容。
取出所有行中的第0列和第1列。
把右上角的 0.9 转换成普通 Python 数字。


digit[0].shape = (3)
digit[:, 0:1].shape = (2,1)
"""
print(digit[-1, 0])
print(digit[0, :])
print(digit[:, 1])
print(digit[:, 0:2])
print(digit[0, -1].item())


In [ ]:
"""
创建 brighter，让所有像素增加 0.1。
创建 darker，让所有像素变成原来的一半。
创建 inverted，让黑白关系反转。
创建一个形状为 [3] 的 Tensor，让第0列加 0.0、第1列加 0.1、第2列加 0.2，并与 digit 相加。
创建一个形状为 [2, 1] 的 Tensor，让第0行加 0.1、第1行加 0.3，并与 digit 相加。
对 pixels 执行归一化，并查看结果的 dtype。

digit + torch.tensor([0.1, 0.2]) 会报错，因为digit shape = (2, 3), 加上shape = (2,) 是对应不上的
"""

brighter = digit + 0.1
darker = digit / 2
inverted = 1 - digit
column_add = torch.tensor([0.0, 0.1, 0.2]) # shape = (3, )
column_add_tensor = digit + column_add # (2, 3) + (3, ) = (2, 3)
row_add = torch.tensor([[0.1], [0.3]]) # shape = (2, 1)
row_add_tensor = digit + row_add # (2, 3) + (2, 1) = (2, 3)

pixels = torch.tensor([[0, 128, 255], [64, 192, 255]]) # shape = (2, 3)
normalized_pixels = pixels / 255 # shape = (2, 3)
print(normalized_pixels)
print(normalized_pixels.dtype) # float32
print(normalized_pixels.shape) # (2,3)


In [ ]:
digit = torch.tensor([[0.0, 0.2, 0.9], [0.7, 0.3, 0.0]])
print(digit.sum())
print(digit.mean())
print(digit.min())
print(digit.max())

In [ ]:
"""
1. 计算 digit 中所有数字的总和及平均值。
2. 计算每一列的平均值。
3. 计算每一行的平均值。
4. 打印上述结果的 shape。
5. 分别运行 mean(dim=1) 和 mean(dim=1, keepdim=True)，比较内容、shape 和 ndim。
6. 创建 row_mean，保存每一行的平均值，并保留维度。
7. 计算 digit - row_mean，观察广播结果。
最后用自己的话解释：为什么 row_mean 使用 [2, 1] 比使用 [2] 更适合与 [2, 3] 的 digit 相减？
"""
digit = torch.tensor([[0.0, 0.2, 0.9], [0.7, 0.3, 0.0]])
print(digit)
mean = digit.mean()
print("Overall mean: ", mean)
column_mean_answer = digit.mean(dim=0)
print("Column mean: ", column_mean_answer)
print("Column mean shape: ", column_mean_answer.shape) # (3,)
row_mean_answer = digit.mean(dim=1)
print("Row mean: ", row_mean_answer)
print("Row mean shape: ", row_mean_answer.shape) # (2,)

row_mean = digit.mean(dim=1, keepdim=True)
print("Row mean keep dim: ", row_mean) # shape = (2, 1)

print(digit - row_mean) # shape: (2, 3) - (2, 1) => (2, 3)

# print(digit - row_mean_answer) # shape:  (2, 3) - (2, ) => error

In [ ]:
"""
1. 打印 digit 的原始形状和元素总数。
2. 把它改变成 [3,2]，保存为 reshaped。
3. 把它展平成 [6]，保存为 flattened。
4. 在最前面添加通道维度，得到 [1,2,3]。
5. 再添加批次维度，得到 [1,1,2,3]。
6. 删除批次维度，恢复成 [1,2,3]。
7. 打印每一步的 shape 和 numel()。
最后回答：
- 为什么改变 shape 后，numel() 始终没有变化？=> 因为shape调整的只是形状，改变不了个数
- reshape(3,2) 是不是等同于把原 Tensor 的行和列互换？先观察具体数字，再作答。=> 并不是，而是先根据先flatten，再根据第一个（3）来进行排组
"""
digit = torch.tensor([[0.0, 0.2, 0.9], [0.7, 0.3, 0.0]])
print("digit: ", digit)
print("Original shape: ", digit.shape, "\nOriginal numel: ", digit.numel())
reshaped = digit.reshape(3, 2)
print("Reshaped digit: ", reshaped)
flattened = digit.reshape(-1) # or digit.flatten()
print("Flattened digit: ", flattened)

channel_digit = digit.unsqueeze(0)
print("Channel digit: ", channel_digit) # shape = (1, 2, 3), numel = 6

batch_digit = channel_digit.unsqueeze(0)
print("Batch digit:", batch_digit) # shape = (1, 1, 2, 3), numel = 6

delete_batch_dim_digit = batch_digit.squeeze(0)
print("Delete batch dim digit: ", delete_batch_dim_digit) # shape = (1, 2, 3), numel = 6


In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = w * 3 + b
y.backward()
print(w.grad)
print(b.grad)

In [ ]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 3
y.backward()
print(x.grad)

In [ ]:
# single
x = torch.tensor(3.0)
target = torch.tensor(7.0)
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.01)
count = 0
latest_formula = f"{w.item():.2f}x + {b.item():.2f}"
print("Latest formula: ", latest_formula)

for i in range(20):
    count += 1
    print("\ncount: ", count)
    optimizer.zero_grad() # 清空旧梯度
    prediction = x * w + b
    loss = (prediction - target) ** 2 # 计算损失
    print("loss amount: " + str(loss.item()))
    loss.backward() # 反向传播
    print("w grad: ", w.grad.item(), "\nb grad: ", b.grad.item())
    optimizer.step() # 更新梯度
    print("updated w: ", w.item(), "\nupdated b: ", b.item())
    latest_formula = f"{w.item():.2f}x + {b.item():.2f}"
    print("Latest formula: ", latest_formula)


In [ ]:
# multiple = actual deep learning
x = torch.tensor([0.0, 1.0, 2.0, 3.0, 4.0])
target = torch.tensor([1.0, 3.0, 5.0, 7.0, 9.0])
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.05)
history = [(0, w.item(), b.item())] # store every count: w and b

total_steps = 30
loop = 5

for step in range(1, total_steps + 1):
    optimizer.zero_grad()
    prediction = x * w + b
    loss = ((prediction - target) ** 2).mean()
    loss.backward()
    optimizer.step()

    if step % loop == 0:
        history.append((step, w.item(), b.item()))
        latest_formula = f"{w.item():.2f}x + {b.item():.2f}"
        print(f"Count {step} latest formula: ", latest_formula)
    

# print("\n\nHistory: ", history)

# plot
import matplotlib.pyplot as plt
plot_x = torch.linspace(0, 5, 100)
plt.figure(figsize=(10, 6))
plt.scatter(x.numpy(), target.numpy(), color="black", label="Training data")
correct_y = 2 * plot_x + 1
plt.plot(
    plot_x.numpy(),
    correct_y.numpy(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Correct: y = 2x + 1"
)
for saved_count, saved_w, saved_b in history:
    plot_y = saved_w * plot_x + saved_b
    color = plt.cm.viridis(saved_count / total_steps)
    label = f"Step {saved_count}" if saved_count in range(0, total_steps + 1, loop) else None
    plt.plot(
        plot_x.numpy(),
        plot_y.numpy(),
        color=color,
        alpha=0.6,
        label=label
    )

plt.xlabel("x")
plt.ylabel("y")
plt.title("How the model learns y = 2x + 1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
"""
1. 创建 [41,1] 的输入 x。
2. 创建 target = x²。
3. 创建 1 → 8 → 1 的神经网络。
4. 打印模型结构。
5. 打印所有参数的名称和 shape。
6. 执行一次 prediction = model(x)。
7. 打印 x、target、prediction 的 shape。
8. 计算初始损失：
loss_fn = nn.MSELoss()
loss = loss_fn(prediction, target)
"""
import torch.nn as nn
x = torch.linspace(-1, 1, 41) # generate 41 numbers starting from -1 to 1 with equal space = [-1, -0.95, -0.9, ..., 1]
x = x.unsqueeze(1) # unsqueeze x to (41, 1), meaning 41 samples each with 1 feature
target = x ** 2 # shape = (41, 1)
model = nn.Sequential(
    nn.Linear(1, 8), # input = 1 feature, output = 8 features, shape => (41, 8) each features have w and b.
    nn.ReLU(),
    nn.Linear(8, 1) # input = 8 features, output = 1 feature, shape => (41, 1)
)

print(model, "\n")
for name, parameter in model.named_parameters():
    print(name, parameter.shape)

prediction = model(x) # shape = (41, 1) => (41, 8) => ReLU => (41, 1)
loss_fn = nn.MSELoss()
loss = loss_fn(prediction, target)
print(loss)

In [ ]:
# ... continue with last cell
import torch.nn as nn
x = torch.linspace(-1, 1, 41)
x = x.unsqueeze(1) # shape = (41, 1)
target = x ** 2 # shape = (41, 1)
model = nn.Sequential(
    nn.Linear(1, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
) # model(x) shape = (41, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) # model.parameters() similar to [w, b]
loss_fn = nn.MSELoss()
loss_history = [] # record each epoch's loss

model.train() # start training
for epoch in range(1000): # reset optimizer -> new prediction using updated parameters -> calculate loss -> loss backward -> optimizer update
    optimizer.zero_grad()
    prediction = model(x)
    loss = loss_fn(prediction, target)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}: loss = {loss.item()}")

model.eval() # end training
import matplotlib.pyplot as plt
plot_x = torch.linspace(-1.0, 1.0, 200).unsqueeze(1) # use 200 for plotting smoothness
with torch.no_grad():
    plot_prediction = model(plot_x) # use parameters trained in the model, without updating parameters.
    correct_y = plot_x ** 2

    plt.scatter(
    x.squeeze(1).numpy(),
    target.squeeze(1).numpy(),
    color="black",
    label="Training data"
    )
    
    plt.plot(
    plot_x.squeeze(1).numpy(),
    correct_y.squeeze(1).numpy(),
    color="red",
    linestyle="--",
    label="Correct: y = x²"
    )

    plt.plot(
    plot_x.squeeze(1).numpy(),
    plot_prediction.squeeze(1).numpy(),
    color="blue",
    label="Neural network"
    )

    # plot loss history
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Neural network learning y = x²")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    plt.plot(loss_history)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training loss")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# Classification example: use nn to classify points to two classes
torch.manual_seed(42)
features = torch.rand(400, 2) # 400 samples with 2 features, each feature is between (0, 1)
features = features * 2 - 1 # make each feature between (-1, 1)

# set rule: x₁² + x₂² < 0.5 belongs to class 1 (153 points), x₁² + x₂² > 0.5 belongs to class 0 (247 points)
inside_circle = features[:, 0] ** 2 + features[:, 1] ** 2 < 0.5 # boolean tensor, shape = (400,)
labels = inside_circle.float().unsqueeze(1) # True = 1.0, False = 0.0, unsqueeze to shape = (400, 1)
class_one = labels.squeeze(1) == 1

# plot actual data
plt.scatter(
    features[~class_one, 0].numpy(), # x1-coordinate with label "not class_one"
    features[~class_one, 1].numpy(), # x2-coordinate with label "not class_one"
    color="blue",
    label=f"Class 0 ({(~class_one).sum().item()} points)", 
    alpha=0.6
)
plt.scatter(
    features[class_one, 0].numpy(), # x1-coordinate with label "class_one"
    features[class_one, 1].numpy(), # x2-coordinate with label "class_one"
    color="red",
    label=f"Class 1 ({class_one.sum().item()} points)", 
    alpha=0.6
)
plt.xlabel("x₁")
plt.ylabel("x₂")
plt.title("Binary classification data")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.axis("equal")
plt.show()


# divide train and test datasets
train_features = features[:320]
train_labels = labels[:320]
test_features = features[320:]
test_labels = labels[320:]

# Build Binary Classification Neural Network
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss() # logits -> 内部进行稳定形式的Sigmoid计算二元交叉熵 -> loss

# train
model.train()
for epoch in range(1000):
    optimizer.zero_grad()
    train_logits = model(train_features) # logits means small or large digits. shape = (320, 1)
    loss = loss_fn(train_logits, train_labels) # we are using logits (number between small and large) to compare with labels (0 or 1)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        with torch.no_grad():
            train_predictions = (train_logits >= 0).float()
            train_accuracy = (
                train_predictions == train_labels
            ).float().mean()

        print(
            f"Epoch {epoch + 1}, "
            f"loss={loss.item():.4f}, "
            f"accuracy={train_accuracy.item():.2%}"
        )

# evaluation
model.eval()
with torch.no_grad():
    test_logits = model(test_features)
    test_loss = loss_fn(test_logits, test_labels)

    test_predictions = (test_logits >= 0).float()

    test_accuracy = (
        test_predictions == test_labels
    ).float().mean()
print("Test loss:", test_loss.item())
print(f"Test accuracy: {test_accuracy.item():.2%}")


# draw model learned boundary
grid_min = -1.1
grid_max = 1.1
grid_size = 200

grid_axis = torch.linspace(
    grid_min,
    grid_max,
    grid_size
)
grid_x1, grid_x2 = torch.meshgrid(
    grid_axis,
    grid_axis,
    indexing="xy"
)
grid_points = torch.stack(
    [
        grid_x1.reshape(-1),
        grid_x2.reshape(-1)
    ],
    dim=1
)
with torch.no_grad():
    grid_logits = model(grid_points)
    # 小于0.5：模型倾向类别0
    # 大于0.5：模型倾向类别1
    grid_probabilities = torch.sigmoid(grid_logits)
logit_map = grid_logits.reshape(grid_x1.shape) # [40000, 1] -> [200, 200]
probability_map = grid_probabilities.reshape(grid_x1.shape)
fig, ax = plt.subplots(figsize=(9, 7))
background = ax.contourf(
    grid_x1.numpy(),
    grid_x2.numpy(),
    probability_map.numpy(),
    levels=20,
    cmap="coolwarm",
    alpha=0.5
)
fig.colorbar(
    background,
    ax=ax,
    label="Probability of class 1"
)
learned_boundary = ax.contour(
    grid_x1.numpy(),
    grid_x2.numpy(),
    logit_map.numpy(),
    levels=[0.0],
    colors="black",
    linewidths=2.5
)
radius = 0.5 ** 0.5
theta = torch.linspace(
    0,
    2 * torch.pi,
    300
)
circle_x1 = radius * torch.cos(theta)
circle_x2 = radius * torch.sin(theta)
ax.plot(
    circle_x1.numpy(),
    circle_x2.numpy(),
    color="green",
    linestyle="--",
    linewidth=2.5,
    label="True boundary: x₁² + x₂² = 0.5"
)
class_one = labels.squeeze(1) == 1
class_zero = ~class_one
ax.scatter(
    features[class_zero, 0].numpy(),
    features[class_zero, 1].numpy(),
    color="blue",
    edgecolors="white",
    linewidths=0.3,
    s=22,
    alpha=0.75,
    label="Class 0"
)
ax.scatter(
    features[class_one, 0].numpy(),
    features[class_one, 1].numpy(),
    color="red",
    edgecolors="white",
    linewidths=0.3,
    s=22,
    alpha=0.75,
    label="Class 1"
)
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
ax.set_title("Learned boundary compared with true boundary")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(grid_min, grid_max)
ax.set_ylim(grid_min, grid_max)
ax.grid(alpha=0.2)
ax.legend(
    bbox_to_anchor=(1.25, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

In [ ]:
# Classification example: three classes
torch.manual_seed(55)
# Create three different classes
class_0 = torch.randn(100, 2) * 0.25 + torch.tensor([-1.0, 0.0]) # shape = (100, 2)
class_1 = torch.randn(100, 2) * 0.25 + torch.tensor([1.0, 0.0]) # shape = (100, 2)
class_2 = torch.randn(100, 2) * 0.25 + torch.tensor([0.0, 1.0]) # shape = (100, 2)
features = torch.cat(
    [class_0, class_1, class_2],
    dim=0
) # shape = (300, 2)

# set labels, each number (0, 1, 2) representing each class
labels_0 = torch.zeros(100, dtype=torch.long) # shape = (100,)
labels_1 = torch.ones(100, dtype=torch.long) # shape = (100,)
labels_2 = torch.full(
    (100,),
    2,
    dtype=torch.long
) # shape = (100,)
labels = torch.cat([labels_0, labels_1, labels_2]) # shape = (300, )

# mix classes and labels using same method
order = torch.randperm(300)
features = features[order]
labels = labels[order]
# separate train data (240) and test data (60)
train_features = features[:240]
train_labels = labels[:240]
test_features = features[240:]
test_labels = labels[240:]

model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 3) # 3 features = 第0个logit → 类别0, 第1个logit → 类别1, 第2个logit → 类别2
)

logits = model(train_features) # shape = (240, 3)
# predictions = logits.argmax(dim=1) # find the max logit to determine what label should be put. # shape = (240,)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, train_labels)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(100):
    optimizer.zero_grad()
    logits = model(train_features)
    loss = loss_fn(logits, train_labels)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            current_logits = model(train_features)
            train_predictions = current_logits.argmax(dim=1) # shape = (240,)
            train_accuracy = (train_predictions == train_labels).float().mean()
        print(
            f"Epoch {epoch + 1}, "
            f"loss={loss.item():.4f}, "
            f"accuracy={train_accuracy.item():.2%}"
        )

model.eval()
with torch.no_grad():
    test_logits = model(test_features)
    test_loss = loss_fn(test_logits, test_labels)
    test_predictions = test_logits.argmax(dim=1)
    test_accuracy = (test_predictions == test_labels).float().mean()
print(f"\nTest result:\nloss={test_loss.item():.4f},\naccuracy={test_accuracy.item():.2%}")